# Redrob — GPU Pre-computation Notebook

Run cells **top to bottom**. Only file to upload: `candidates.jsonl` (~465 MB).

**This notebook produces 3 files that auto-download at the end:**
- `candidate_embeddings.npy`
- `jd_embedding.npy`
- `candidate_features.pkl`

Copy all 3 into your local `cache/` folder, then run `rank.py` locally.

In [1]:
!pip install sentence-transformers tqdm numpy -q
import torch
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("VRAM:", round(mem, 1), "GB")
else:
    print("No GPU — go to Runtime > Change runtime type > T4 GPU > Save, then restart")


PyTorch: 2.11.0+cu128
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
%%writefile honeypot.py
from datetime import datetime


def _education_timeline_valid(candidate):
    edu = candidate.get("education", [])
    degrees = sorted(edu, key=lambda e: e.get("start_year", 9999))
    bachelors_end = None
    for deg in degrees:
        level = deg.get("degree", "").lower()
        if any(b in level for b in ["b.tech","b.e","b.sc","be","btech","b.com"]):
            bachelors_end = deg.get("end_year")
        elif any(m in level for m in ["m.tech","m.e","m.sc","me","mtech","mba","m.s"]):
            masters_start = deg.get("start_year", 9999)
            if bachelors_end and masters_start < bachelors_end:
                return False
    return True


def detect_honeypot(candidate):
    flags = 0
    skills   = candidate.get("skills", [])
    career   = candidate.get("career_history", [])
    profile  = candidate.get("profile", {})
    signals  = candidate.get("redrob_signals", {})
    assessed = signals.get("skill_assessment_scores", {})
    for skill in skills:
        if skill.get("proficiency") in ["expert","advanced"] and skill.get("duration_months",1) == 0:
            flags += 2
            break
    expert_count = sum(1 for s in skills if s.get("proficiency") == "expert")
    if expert_count > 8:
        flags += 2
    stuffing = sum(
        1 for s in skills
        if assessed.get(s.get("name","")) is not None
        and s.get("proficiency") in ["advanced","expert"]
        and assessed[s["name"]] < 40
    )
    flags += min(stuffing, 3)
    yoe = float(profile.get("years_of_experience", 0))
    total_months = sum(j.get("duration_months", 0) for j in career)
    if yoe > 2 and total_months < (yoe * 12 * 0.45):
        flags += 2
    for job in career:
        if job.get("duration_months", 0) > 130:
            flags += 1
    if not _education_timeline_valid(candidate):
        flags += 2
    current_title = profile.get("current_title","").lower()
    non_tech = ["marketing","sales","hr ","operations manager","finance","mechanical engineer","brand manager"]
    ai_set = {"ml","nlp","llm","embedding","deep learning","rag","retrieval"}
    has_ai = any(kw in s.get("name","").lower() for s in skills for kw in ai_set)
    if has_ai and any(t in current_title for t in non_tech):
        flags += 3
    if flags == 0: return 1.00
    if flags == 1: return 0.85
    if flags == 2: return 0.65
    if flags == 3: return 0.40
    if flags == 4: return 0.20
    return 0.05


Writing honeypot.py


In [3]:
%%writefile constants.py
CONSULTING_FIRMS = {
    "tcs","tata consultancy","wipro","infosys","accenture",
    "cognizant","capgemini","hcl","tech mahindra","mindtree",
    "mphasis","hexaware","ltimindtree","l&t infotech","birlasoft",
}


def is_consulting(company_name):
    return any(firm in company_name.lower() for firm in CONSULTING_FIRMS)


Writing constants.py


In [4]:
%%writefile features.py
import math
from datetime import datetime, date
from honeypot import detect_honeypot
from constants import is_consulting

PREFERRED_LOCATIONS = {
    "pune":1.00,"noida":1.00,"hyderabad":0.90,"mumbai":0.90,
    "delhi":0.90,"gurugram":0.90,"gurgaon":0.90,"new delhi":0.90,
    "bangalore":0.75,"bengaluru":0.75,"chennai":0.70,"kolkata":0.65,
}
JD_SKILLS = {
    "faiss","milvus","pinecone","weaviate","qdrant","opensearch",
    "elasticsearch","pgvector","embeddings","sentence-transformers",
    "dense retrieval","hybrid search","information retrieval","bm25",
    "llms","fine-tuning llms","lora","qlora","peft","rag","ranking",
    "recommendation systems","learning to rank","haystack","langchain",
    "nlp","transformers","hugging face","hugging face transformers",
    "pytorch","python","scikit-learn","mlops","ndcg","evaluation",
}
CV_SKILLS = {
    "computer vision","image classification","object detection","yolo",
    "resnet","cnn","image segmentation","ocr","tts","speech recognition",
    "gans","diffusion models",
}


def _days_since(date_str):
    try:
        d = datetime.strptime(date_str, "%Y-%m-%d").date()
        return (date.today() - d).days
    except Exception:
        return 999


def career_quality(candidate):
    career  = candidate.get("career_history", [])
    profile = candidate.get("profile", {})
    skills  = candidate.get("skills", [])
    if not career: return 0.10
    if all(is_consulting(j.get("company","")) for j in career): return 0.05
    score = 0.40
    product_jobs = [j for j in career if not is_consulting(j.get("company",""))]
    score += 0.20 * (len(product_jobs) / len(career))
    ai_kw = ["ml","ai","machine learning","nlp","data scientist","search","ranking","retrieval","recommendation"]
    ai_jobs = [j for j in career if any(kw in j.get("title","").lower() for kw in ai_kw)]
    score += 0.15 * (len(ai_jobs) / len(career))
    yoe = float(profile.get("years_of_experience", 0))
    if 4 <= yoe <= 10: score += 0.10
    elif 3 <= yoe < 4 or 10 < yoe <= 12: score += 0.05
    tenures = [j.get("duration_months",0) for j in career]
    avg_t = sum(tenures)/len(tenures) if tenures else 0
    if avg_t >= 24: score += 0.10
    elif avg_t >= 18: score += 0.05
    elif avg_t < 12: score -= 0.15
    all_names = [s.get("name","").lower() for s in skills]
    cv_count = sum(1 for n in all_names if n in CV_SKILLS)
    if all_names and (cv_count/len(all_names)) > 0.40: score -= 0.12
    wrong = ["marketing manager","sales manager","hr manager","operations manager","brand manager","mechanical engineer"]
    cur = profile.get("current_title","").lower()
    if any(d in cur for d in wrong): score -= 0.30
    return max(0.0, min(1.0, score))


def availability(candidate):
    s = candidate.get("redrob_signals", {})
    days_idle = _days_since(s.get("last_active_date","2020-01-01"))
    recency   = math.exp(-days_idle * math.log(2) / 90)
    openness  = 1.0 if s.get("open_to_work_flag", False) else 0.55
    response  = s.get("recruiter_response_rate", 0.0)
    nd = s.get("notice_period_days", 90)
    notice = 1.00 if nd<=30 else 0.75 if nd<=60 else 0.50 if nd<=90 else 0.20
    interview = s.get("interview_completion_rate", 0.5)
    return recency*0.30 + openness*0.20 + response*0.25 + notice*0.15 + interview*0.10


def location(candidate):
    profile = candidate.get("profile", {})
    signals = candidate.get("redrob_signals", {})
    loc = profile.get("location","").lower()
    country = profile.get("country","").lower()
    can_relocate = signals.get("willing_to_relocate", False)
    for city, city_score in PREFERRED_LOCATIONS.items():
        if city in loc: return city_score
    if country == "india" or "india" in loc:
        return 0.65 if can_relocate else 0.50
    return 0.35 if can_relocate else 0.10


def skill_depth(candidate):
    skills   = candidate.get("skills", [])
    signals  = candidate.get("redrob_signals", {})
    assessed = signals.get("skill_assessment_scores", {})
    gh_raw   = signals.get("github_activity_score", -1)
    gh_score = gh_raw/100 if gh_raw >= 0 else 0.15
    if not skills: return gh_score * 0.25
    total, count = 0.0, 0
    prof_map = {"beginner":0.20,"intermediate":0.50,"advanced":0.80,"expert":1.00}
    for skill in skills:
        name = skill.get("name","").lower()
        if not any(kw in name for kw in JD_SKILLS): continue
        count += 1
        prof = skill.get("proficiency","beginner")
        dur  = skill.get("duration_months", 0)
        base = prof_map.get(prof, 0.20)
        dur_s = min(dur/36, 1.0)
        av = assessed.get(skill.get("name",""))
        if av is not None:
            verified = av/100
            if prof in ["advanced","expert"] and av < 50: verified *= 0.50
            contrib = verified*0.70 + dur_s*0.30
        else:
            contrib = base*0.60 + dur_s*0.40
        total += contrib
    if count == 0: return gh_score * 0.25
    return (total/count)*0.75 + gh_score*0.25


def extract_features(candidate):
    return {
        "candidate_id":        candidate["candidate_id"],
        "career_quality":      career_quality(candidate),
        "availability":        availability(candidate),
        "location":            location(candidate),
        "skill_depth":         skill_depth(candidate),
        "honeypot_multiplier": detect_honeypot(candidate),
    }


Writing features.py


In [5]:
from google.colab import files
import os
print("Upload candidates.jsonl (~465 MB). Takes 2-5 min on most connections.")
uploaded = files.upload()
if os.path.exists("candidates.jsonl"):
    size = os.path.getsize("candidates.jsonl") / 1e6
    print("Ready:", round(size), "MB")
else:
    print("Please upload candidates.jsonl")


Upload candidates.jsonl (~465 MB). Takes 2-5 min on most connections.


KeyboardInterrupt: 

In [ ]:
import json, os, pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from features import extract_features
import torch

CACHE_DIR  = "cache"
MODEL_NAME = "BAAI/bge-base-en-v1.5"
BATCH_SIZE = 512
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

JD_QUERY = (
    "Senior AI engineer 5-9 years at product companies not IT services. "
    "Production embeddings retrieval sentence-transformers BGE E5. "
    "Vector databases hybrid search Pinecone Weaviate Qdrant Milvus FAISS Elasticsearch. "
    "Shipped ranking search recommendation system real users at scale. "
    "Strong Python production ML not pure research. "
    "Evaluation frameworks NDCG MRR MAP AB testing offline online correlation. "
    "LLM fine-tuning LoRA QLoRA PEFT. "
    "NLP information retrieval not computer vision speech. "
    "Pune Noida Hyderabad Mumbai Delhi NCR India."
)


def build_text(c):
    p  = c.get("profile", {})
    ca = c.get("career_history", [])
    sk = c.get("skills", [])
    si = c.get("redrob_signals", {})
    av = si.get("skill_assessment_scores", {})
    parts = [
        p.get("current_title","") + " at " + p.get("current_company","") + ". " +
        str(p.get("years_of_experience",0)) + " yrs. Industry: " + p.get("current_industry","")
    ]
    for job in ca[:4]:
        d = job.get("description","").strip()
        if d:
            parts.append("At " + job.get("company","") + " as " + job.get("title","") + ": " + d)
    for s in sk:
        nm = s.get("name","")
        pr = s.get("proficiency","")
        du = s.get("duration_months",0)
        aval = av.get(nm)
        if pr in ["advanced","expert"] and du >= 12:
            if aval and aval >= 60:
                parts.append("Verified " + nm + ": " + str(round(aval)) + "/100, " + str(du) + " months.")
            elif not aval:
                parts.append("Advanced " + nm + ", " + str(du) + " months.")
    sm = p.get("summary","").strip()
    if sm: parts.append(sm)
    return " ".join(parts)


os.makedirs(CACHE_DIR, exist_ok=True)
print("Loading model:", MODEL_NAME)
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

print("Loading candidates...")
candidates = []
with open("candidates.jsonl", "rt", encoding="utf-8") as f:
    for line in tqdm(f, desc="Parsing"):
        line = line.strip()
        if line: candidates.append(json.loads(line))
print(str(len(candidates)) + " candidates loaded")

print("Building texts...")
texts = [build_text(c) for c in tqdm(candidates, desc="Texts")]

print("Embedding on " + DEVICE + "...")
embs = model.encode(
    texts, batch_size=BATCH_SIZE, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True
)

print("Embedding JD...")
jd_emb = model.encode(
    "Represent this sentence for searching relevant passages: " + JD_QUERY,
    normalize_embeddings=True, convert_to_numpy=True
)

print("Extracting features...")
features_list = [extract_features(c) for c in tqdm(candidates, desc="Features")]

np.save(CACHE_DIR + "/candidate_embeddings.npy", embs)
np.save(CACHE_DIR + "/jd_embedding.npy", jd_emb)
with open(CACHE_DIR + "/candidate_features.pkl", "wb") as fh:
    pickle.dump({"candidates": candidates, "features": features_list}, fh, protocol=4)

print("\nDone! Shape:", embs.shape)
print("Cache saved to", CACHE_DIR)


In [ ]:
from google.colab import files
print("Downloading 3 cache files to your computer...")
print("Accept each download prompt when it appears.")
print()
files.download("cache/candidate_embeddings.npy")
files.download("cache/jd_embedding.npy")
files.download("cache/candidate_features.pkl")
print("\nAll done!")
print("Copy the 3 files into your local cache/ folder, then run:")
print("  python3 rank.py --candidates candidates.jsonl --out submission.csv")
